In [1]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
valid_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][10]

'Lucas: Hey! How was your day?\r\nDemi: Hey there! \r\nDemi: It was pretty fine, actually, thank you!\r\nDemi: I just got promoted! :D\r\nLucas: Whoa! Great news!\r\nLucas: Congratulations!\r\nLucas: Such a success has to be celebrated.\r\nDemi: I agree! :D\r\nDemi: Tonight at Death & Co.?\r\nLucas: Sure!\r\nLucas: See you there at 10pm?\r\nDemi: Yeah! See you there! :D'

In [5]:
train_data.shape

(14732, 3)

In [6]:
valid_data.shape

(818, 3)

In [7]:
# Random Sampling 
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
valid_data = valid_data.sample(n=500, random_state=42).reset_index(drop=True)

In [8]:
train_data.shape

(4000, 3)

### Data Preprocessing

In [9]:
import re 

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # Replace the lines
    text = re.sub(r"\s+", " ", text) # Removw extra spaces
    text = re.sub(r"<.*?>", " ", text) # Remove html tags
    text = text.strip().lower()  # Remove extra spaces from start and end
    return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

valid_data["dialogue"] = valid_data["dialogue"].apply(clean_data)
valid_data["summary"] = valid_data["summary"].apply(clean_data)

In [11]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenize

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [13]:
# Raw data => tokenized inputs for fine-tuning

def tokenize(data): 
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = target["input_ids"]  # Token ids => add to input as labels
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
valid_dataset = valid_data.apply(tokenize, axis=1).tolist()

In [15]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# Input ids = dialouge => token ids
# Attention mask = 1=> valid tokenize value, 0=> not valid 
# Labels = target => summary token
# 1 = end of sequence
# 0 = extra padding value

In [16]:
len(train_dataset[0]["input_ids"])

512

In [17]:
len(train_dataset[0]["labels"])

150

### Model

In [18]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [19]:
# Fine-tune 
import torch 

if torch.backends.mps.is_available(): 
    device = torch.device("mps")
elif torch.cuda.is_available(): 
    device = torch.device("cuda")
else: 
    device = torch.device("cpu")

print("device : ", device)
model.to(device)

device :  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training arguments 

train_agrs = TrainingArguments(
    output_dir = "./results", 

    num_train_epochs=6, 
    weight_decay=0.01, 

    per_device_eval_batch_size=8, 
    per_device_train_batch_size=8, 

    eval_strategy="epoch", 
    save_strategy="epoch", 

    warmup_steps=500
)

In [21]:
trainer = Trainer(
    model=model, 
    args=train_agrs, 
    train_dataset=train_dataset, 
    eval_dataset=valid_dataset
)

In [22]:
# Model Training 
trainer.train()

C:\Users\mausa\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.624489,0.380083
2,0.396790,0.359977
3,0.374074,0.354027
4,0.363074,0.350865
5,0.356373,0.349925
6,0.351206,0.349048


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\mausa\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

C:\Users\mausa\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\mausa\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9110009511311848, metrics={'train_runtime': 70200.1924, 'train_samples_per_second': 0.342, 'train_steps_per_second': 0.043, 'total_flos': 3248203235328000.0, 'train_loss': 0.9110009511311848, 'epoch': 6.0})

In [23]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [25]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Test the core logic for summarization

In [26]:
def summarize_dialogue(dialogue):
    # Cleaning
    dialogue = clean_data(dialogue)

    # Tokenize 
    inputs = tokenizer(
        dialogue, 
        padding="max_length", 
        max_length=512, 
        truncation=True, 
        return_tensors="pt"
    )

    # Generate the summary => Token Ids 
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150, 
        num_beams=4,
        early_stopping=True
    )

    # Token IDs converted into summary => Decoding 
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

In [27]:
test_dialogue = """Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way."""

summary = summarize_dialogue(test_dialogue)

print("summary : ", summary)

summary :  ai technology continues to expand rapidly across industries, from healthcare to finance and education. ai adoption has significantly increased over the past few years.
